In [13]:
from alazar import ATSApi, open_ats9371
%matplotlib qt
api = ATSApi()
board = open_ats9371(api, system_id=1, board_id=1)

print(board)

BoardInfo(handle=1356, kind=33, memory_samples=4294967294, bits_per_sample=12)


In [14]:
import ctypes
import time
from contextlib import suppress

import matplotlib.pyplot as plt
import numpy as np

from alazar.ats_api import ATSApi
from alazar.ats9371 import (
    DmaBuffer,
    TriggerConfig,
    allocate_dma_memory,
    bytes_per_sample,
    configure_ats9371,
    open_ats9371,
    release_dma_memory,
    unpack_adc_codes,
)
from alazar.constants import (
    ADMA_EXTERNAL_STARTCAPTURE,
    ADMA_NPT,
    CHANNEL_A,
)

# ===== 設定 =====
SAMPLE_RATE_HZ = 1e9
SAMPLES_PER_RECORD = 2560
RECORDS_PER_BUFFER = 100
DMA_BUFFER_COUNT = 4
BUFFER_TIMEOUT_MS = 5000
PLOT_INTERVAL_S = 0.05
RECORDS_PER_ACQUISITION = 0x7FFFFFFF

api = ATSApi(r"C:\Windows\System32\ATSApi.dll")
board = open_ats9371(api, system_id=1, board_id=1)

# timeout_ticks=1：自動 trigger，不等 external trigger
configure_ats9371(
    api,
    board,
    TriggerConfig(
        level=128,
        delay_samples=0,
        timeout_ticks=1,
    ),
)

sample_count = SAMPLES_PER_RECORD * RECORDS_PER_BUFFER
size_bytes = sample_count * bytes_per_sample(board.bits_per_sample)
buffers = []

try:
    # 建立 DMA buffers
    for _ in range(DMA_BUFFER_COUNT):
        address = allocate_dma_memory(size_bytes)

        buffers.append(
            DmaBuffer(
                address=address,
                sample_count=sample_count,
                size_bytes=size_bytes,
            )
        )

    api.set_record_size(
        board.handle,
        0,
        SAMPLES_PER_RECORD,
    )

    api.before_async_read(
        board.handle,
        CHANNEL_A,
        0,
        SAMPLES_PER_RECORD,
        RECORDS_PER_BUFFER,
        RECORDS_PER_ACQUISITION,
        ADMA_NPT | ADMA_EXTERNAL_STARTCAPTURE,
    )

    for buffer in buffers:
        api.post_async_buffer(
            board.handle,
            buffer.address,
            buffer.size_bytes,
        )

    # ===== 建立 Live Plot =====
    time_us = np.arange(SAMPLES_PER_RECORD) / SAMPLE_RATE_HZ * 1e6

    adc_midpoint = float(1 << (board.bits_per_sample - 1))
    volts_per_code = 0.4 / adc_midpoint

    plt.ion()

    fig, ax = plt.subplots(figsize=(10, 5))

    line, = ax.plot(
        time_us,
        np.zeros(SAMPLES_PER_RECORD),
    )

    ax.set_xlabel("Time (us)")
    ax.set_ylabel("Voltage (V)")
    ax.set_xlim(time_us[0], time_us[-1])
    # ax.set_ylim(-0.4, 0.4)
    ax.grid(True)

    plt.show(block=False)

    # ===== 開始擷取 =====
    api.start_capture(board.handle)

    print("開始持續擷取，按 Ctrl+C 或關閉視窗停止")

    completed_buffers = 0
    last_plot_time = 0.0

    while plt.fignum_exists(fig.number):
        buffer = buffers[completed_buffers % DMA_BUFFER_COUNT]

        api.wait_async_buffer(
            board.handle,
            buffer.address,
            BUFFER_TIMEOUT_MS,
        )

        # 將 DMA memory 轉成 NumPy array
        array_type = ctypes.c_uint16 * buffer.sample_count

        dma_view = np.ctypeslib.as_array(
            array_type.from_address(buffer.address)
        )

        raw_dma = dma_view.copy().reshape(
            RECORDS_PER_BUFFER,
            SAMPLES_PER_RECORD,
        )

        # 顯示目前 buffer 最後一筆 waveform
        raw_adc = unpack_adc_codes(
            raw_dma[-1],
            board.bits_per_sample,
        )

        # 立即將 DMA buffer 放回 queue
        api.post_async_buffer(
            board.handle,
            buffer.address,
            buffer.size_bytes,
        )

        completed_buffers += 1
        now = time.monotonic()

        if now - last_plot_time >= PLOT_INTERVAL_S:
            voltage = (
                raw_adc.astype(np.float64) - adc_midpoint
            ) * volts_per_code

            line.set_ydata(voltage)

            total_records = completed_buffers * RECORDS_PER_BUFFER
            ax.set_title(
                f"ATS9371 Live Plot - {total_records:,} records"
            )

            fig.canvas.draw_idle()
            fig.canvas.flush_events()
            plt.pause(0.001)

            last_plot_time = now

except KeyboardInterrupt:
    print("\n停止擷取")

finally:
    with suppress(Exception):
        api.abort_async_read(board.handle)

    for buffer in buffers:
        with suppress(Exception):
            release_dma_memory(buffer.address)

    plt.close("all")
    print("DMA buffers 已釋放")

開始持續擷取，按 Ctrl+C 或關閉視窗停止

停止擷取
DMA buffers 已釋放


In [ ]:
print(raw_adc)
print(voltage)

plt.figure()
plt.plot(voltage)
plt.show()

[2050 2038 2050 ... 2034 2046 2036]
[ 0.00039063 -0.00195312  0.00039063 ... -0.00273438 -0.00039063
 -0.00234375]


In [ ]:
from awg5200 import (
    AWG5208,
    delay_auto,
    gaussian_square_ns,
    parallel,
    waveform,
)

resource = "TCPIP0::192.168.10.171::inst0::INSTR"
sample_rate_hz = 2.5e9
awg = AWG5208.connect(
    resource,
    timeout_ms=60_000,
)

awg.set_awg_mode()
awg.set_sample_rate(sample_rate_hz)

print("Connected:", awg.identify())




qubit1_envelope = gaussian_square_ns(
    duration_ns=100,
    sample_rate_hz=sample_rate_hz,
    edge_sigma_ns=10,
    amplitude_volts=0.2,
)
qubit2_envelope = gaussian_square_ns(
    duration_ns=300,
    sample_rate_hz=sample_rate_hz,
    edge_sigma_ns=10,
    amplitude_volts=0.2,
)

readout_envelope = gaussian_square_ns(
    duration_ns=1000,
    sample_rate_hz=sample_rate_hz,
    edge_sigma_ns=10,
    amplitude_volts=0.2,
)

qubit1 = waveform(
    qubit1_envelope,
    fc=0,
    ch=2,
    name="qubit1",
)

qubit2 = waveform(
    qubit2_envelope,
    fc=0,
    ch=4,
    name="qubit2",
)


readout = waveform(
    readout_envelope,
    fc=0,
    ch=3,
    name="readout",
)


timeline = (
    parallel(qubit1, qubit2)
    / delay_auto(10e-9)
    / readout
)

names = awg.upload_timeline(
    timeline,
    amplitude_vpp={
        2: 0.5,
        3: 0.5,
        4: 0.5,
    },
    total_duration_s=10e-6,
)


marker_name = awg.marker(
    waveform_ch=3,
    marker_ch=1,
)

# awg.run(wait_until_ready=True)

print("Channel 1 marker:", marker_name)
print("Channel 2 qubit:", names[2])
print("Channel 3 readout:", names[3])
print("Run state:", awg.run_state())
print("Error:", awg.query("SYSTem:ERRor?"))

Connected: TEKTRONIX,AWG5208,B030598,FV:6.6.0131.0
Channel 1 marker: marker_ch1_for_ch3
Channel 2 qubit: qubit1
Channel 3 readout: readout
Run state: 0
Error: 0,"No error"
